In [ ]:
import pandas as pd
import openpyxl
import os
import re
from datetime import datetime
import win32com.client as win32
import win32api

# ==========================
# CONFIGURAÇÕES
# ==========================

CAMINHO_BASE   = r'C:\Users\v.alves\OneDrive - SPOT\Área de Trabalho\ENVIOS PRODUTO FOCO\TESTE\Base_Extracao_PE_Produto_Foco_TESTE.xlsx'
PASTA_SAIDA    = r'C:\Users\v.alves\OneDrive - SPOT\Área de Trabalho\ENVIOS PRODUTO FOCO\Arquivos_Gerados'
ARQUIVO_LOG    = r'C:\Users\v.alves\OneDrive - SPOT\Área de Trabalho\ENVIOS PRODUTO FOCO\LOG_ENVIO_PE.xlsx'

ABA_DADOS      = 'BASE_PE'
ABA_BASE       = 'BASE_PE'
ABA_PIVOT      = 'TT_PE'

# Colunas A até O (15 colunas)
COLUNAS_MODELO = [
    'COD_PESQUISA', 'DATA', 'COD_LOJA', 'COD_SAP', 'NOME_FANTASIA',
    'BANDEIRA', 'DES_REGIAO', 'NOM_PESSOA_COMPLETO', 'DES_CATEGORIA',
    'DES_SUB_CATEGORIA', 'DES_TIPO_PONTO_EXTRA', 'FOTO', 'CHAVE',
    'EXISTE', 'PE_RETORNO'
]

EMAILS_CC      = [
    'coordenador@empresa.com',
    'diretoria@empresa.com'
]

MAX_TENTATIVAS = 2
MODO_TESTE     = True


# ==========================
# HELPERS
# ==========================

def tipo_label(modo_teste):
    return "TESTE" if modo_teste else "PRODUÇÃO"

def normalizar_valor(v):
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(v, pd.Timestamp) and v.tzinfo is not None:
        return v.tz_localize(None)
    return v

def limpar_nome(nome):
    return re.sub(r'[\\/*?:"<>|]', "", str(nome))

def path_curto(caminho):
    return win32api.GetShortPathName(os.path.abspath(caminho))


# ==========================
# PREPARAÇÃO
# ==========================

df = pd.read_excel(CAMINHO_BASE, sheet_name=ABA_DADOS)
os.makedirs(PASTA_SAIDA, exist_ok=True)

data_log  = datetime.now().strftime('%d-%m-%Y %H:%M:%S')
data_nome = datetime.now().strftime('%d-%m-%Y')


# ==========================
# VALIDAÇÃO
# ==========================

def validar_base():
    ausentes = [c for c in ['ESPECIALISTA', 'EMAIL ESPECIALISTA', 'GERENTE', 'EMAIL GERENTE']
                if c not in df.columns]
    if ausentes:
        raise ValueError(f"Colunas ausentes na base: {ausentes}")
    print("✅ Base validada com sucesso.")


# ==========================
# LOG
# ==========================

registros_log = []

def registrar_log(tipo, nome, email, status, erro=""):
    registros_log.append({
        "DATA_ENVIO": data_log, "TIPO": tipo, "NOME": nome,
        "EMAIL": email, "STATUS": status, "ERRO": erro
    })

def salvar_log():
    if not registros_log:
        return
    novo = pd.DataFrame(registros_log)
    if os.path.exists(ARQUIVO_LOG):
        final = pd.concat([pd.read_excel(ARQUIVO_LOG), novo], ignore_index=True)
    else:
        final = novo
    final.to_excel(ARQUIVO_LOG, index=False)
    print(f"\n📋 Log salvo: {len(registros_log)} registros em {ARQUIVO_LOG}")


# ==========================
# GERAR RELATÓRIO
# ==========================

def gerar_relatorio(df_filtrado, nome_saida):
    caminho_final     = os.path.abspath(nome_saida)
    colunas_presentes = [c for c in COLUNAS_MODELO if c in df_filtrado.columns]
    df_para_gravar    = df_filtrado[colunas_presentes].reset_index(drop=True)

    print(f"      → gravando {len(df_para_gravar)} linhas | {os.path.basename(nome_saida)}")

    wb   = openpyxl.Workbook()
    ws_b = wb.active
    ws_b.title = ABA_BASE

    # ── BASE_PE: cabeçalho + dados filtrados (colunas A:O) ──
    for col_idx, col_nome in enumerate(colunas_presentes, start=1):
        ws_b.cell(row=1, column=col_idx, value=col_nome)

    for row_idx, row in enumerate(df_para_gravar.itertuples(index=False), start=2):
        for col_idx, v in enumerate(row, start=1):
            ws_b.cell(row=row_idx, column=col_idx, value=normalizar_valor(v))

    # ── TT_PE: contagem distinta de CHAVE por SUB_CATEGORIA e TIPO_PONTO_EXTRA ──
    ws_t = wb.create_sheet(ABA_PIVOT)

    cols_group = ['DES_SUB_CATEGORIA', 'DES_TIPO_PONTO_EXTRA']
    col_chave  = 'CHAVE'

    cols_disponiveis = [c for c in cols_group if c in df_para_gravar.columns]

    if cols_disponiveis and col_chave in df_para_gravar.columns:
        resumo = (
            df_para_gravar
            .groupby(cols_disponiveis, dropna=False)[col_chave]
            .nunique()
            .reset_index()
            .rename(columns={col_chave: 'QTD_PE'})
            .sort_values(cols_disponiveis)
            .reset_index(drop=True)
        )

        total_pe = int(resumo['QTD_PE'].sum())

        # Cabeçalho
        cabecalho = cols_disponiveis + ['QTD_PE']
        for col_idx, nome_col in enumerate(cabecalho, start=1):
            ws_t.cell(row=1, column=col_idx, value=nome_col)

        # Dados do resumo
        for row_idx, row in enumerate(resumo.itertuples(index=False), start=2):
            for col_idx, v in enumerate(row, start=1):
                ws_t.cell(row=row_idx, column=col_idx, value=normalizar_valor(v))

        # Linha de total
        linha_total = len(resumo) + 2
        ws_t.cell(row=linha_total, column=len(cols_disponiveis),     value='TOTAL')
        ws_t.cell(row=linha_total, column=len(cols_disponiveis) + 1, value=total_pe)

        print(f"      → TT_PE: {len(resumo)} combinações | TOTAL PE: {total_pe}")

    wb.save(caminho_final)
    wb.close()


# ==========================
# ENVIAR E-MAIL
# ==========================

def enviar_email(destinatario, nome, arquivo, tipo, outlook_app):
    if pd.isna(destinatario) or str(destinatario).strip() == "":
        registrar_log(tipo, nome, destinatario, "ERRO", "Sem e-mail")
        return
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            mail         = outlook_app.CreateItem(0)
            mail.To      = str(destinatario).strip()
            mail.CC      = "; ".join(EMAILS_CC)
            mail.Subject = f"Relatório Semanal - Pontos Extras ({data_nome})"
            mail.Body    = (
                f"Olá {nome},\n\n"
                f"Segue relatório atualizado de Pontos Extras ({tipo}).\n\n"
                f"Qualquer dúvida fico à disposição.\n\nAtenciosamente,"
            )
            mail.Attachments.Add(os.path.abspath(arquivo))
            mail.Send()
            registrar_log(tipo, nome, destinatario, "ENVIADO")
            return
        except Exception as e:
            if tentativa < MAX_TENTATIVAS:
                print(f"    ⚠️ Tentativa {tentativa} falhou, tentando novamente...")
            else:
                registrar_log(tipo, nome, destinatario, "ERRO", str(e))
                print(f"    ❌ ERRO: {e}")


# ==========================
# VALIDAÇÕES INICIAIS
# ==========================

validar_base()

print(f"Base carregada: {len(df)} registros | "
      f"{df['ESPECIALISTA'].nunique()} especialistas | "
      f"{df['GERENTE'].nunique()} gerentes")

if not MODO_TESTE:
    total_dest = df['ESPECIALISTA'].nunique() + df['GERENTE'].nunique()
    resposta = input(
        f"\n⚠️  MODO PRODUÇÃO — {total_dest} destinatários. Confirma? [s/n]: "
    ).strip().lower()
    if resposta != 's':
        print("Cancelado.")
        raise SystemExit(0)

if MODO_TESTE:
    print("\n⚠️  MODO TESTE — nenhum e-mail será enviado.\n")
    PASTA_ATUAL = os.path.join(PASTA_SAIDA, 'TESTE')
    os.makedirs(PASTA_ATUAL, exist_ok=True)
else:
    PASTA_ATUAL = PASTA_SAIDA

outlook_app = None if MODO_TESTE else win32.Dispatch('outlook.application')

try:
    # --- ESPECIALISTAS ---
    especialistas = df['ESPECIALISTA'].dropna().unique()
    print(f"Processando {len(especialistas)} especialistas...")

    for especialista in especialistas:
        df_esp       = df[df['ESPECIALISTA'] == especialista].copy()
        email        = df_esp['EMAIL ESPECIALISTA'].iloc[0]
        nome_limpo   = limpar_nome(especialista)
        prefixo      = "TESTE_" if MODO_TESTE else ""
        nome_arquivo = os.path.join(PASTA_ATUAL, f"{prefixo}Especialista_{nome_limpo}_{data_nome}.xlsx")

        print(f"  [{tipo_label(MODO_TESTE)}] {especialista} → {len(df_esp)} registros")
        try:
            gerar_relatorio(df_esp, nome_arquivo)
            if MODO_TESTE:
                registrar_log("Especialista", especialista, email, "GERADO")
                print(f"    ✅ {nome_arquivo}")
            else:
                enviar_email(email, especialista, nome_arquivo, "Especialista", outlook_app)
        except Exception as e:
            registrar_log("Especialista", especialista, email, "ERRO", str(e))
            print(f"    ❌ ERRO: {e}")

    # --- GERENTES ---
    gerentes = df['GERENTE'].dropna().unique()
    print(f"\nProcessando {len(gerentes)} gerentes...")

    for gerente in gerentes:
        df_ger       = df[df['GERENTE'] == gerente].copy()
        email        = df_ger['EMAIL GERENTE'].iloc[0]
        nome_limpo   = limpar_nome(gerente)
        prefixo      = "TESTE_" if MODO_TESTE else ""
        nome_arquivo = os.path.join(PASTA_ATUAL, f"{prefixo}Gerente_{nome_limpo}_{data_nome}.xlsx")

        print(f"  [{tipo_label(MODO_TESTE)}] {gerente} → {len(df_ger)} registros")
        try:
            gerar_relatorio(df_ger, nome_arquivo)
            if MODO_TESTE:
                registrar_log("Gerente", gerente, email, "GERADO")
                print(f"    ✅ {nome_arquivo}")
            else:
                enviar_email(email, gerente, nome_arquivo, "Gerente", outlook_app)
        except Exception as e:
            registrar_log("Gerente", gerente, email, "ERRO", str(e))
            print(f"    ❌ ERRO: {e}")

finally:
    salvar_log()

if MODO_TESTE:
    print(f"\n✅ Modo teste concluído! Arquivos em: {PASTA_ATUAL}")
else:
    print("\n✅ Processo finalizado com sucesso!")